# 01 业务理解与数据入口

这个文件只做项目入口：明确风控问题、确认目标变量、看样本基准、列出下一步必须处理的数据问题。

本文件不做复杂清洗，不做模型。


## 1. 风控业务问题

本项目要解决的问题是：识别未来两年更可能发生严重逾期或财务困境的客户。

实际风控分析中，第一步只需要回答：

1. 坏客户比例是多少？
2. 数据字段是否能支持风险分析？
3. 哪些数据质量问题会影响后续判断？


## 2. 导入工具和设置路径


In [28]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

# 项目根目录：如果你的项目文件夹换位置，只需要改这一行。
PROJECT_ROOT = Path(r"E:\DataAnalysis\DataAnalysisRoad\Projects\01_Credit_Risk_Analysis")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cs-training.csv"

RAW_DATA_PATH


WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/data/raw/cs-training.csv')

## 3. 读取数据并重命名字段

原始字段名较长，不利于分析。这里把字段改成更短、更接近业务含义的名字。


In [29]:
raw_df = pd.read_csv(RAW_DATA_PATH)

# 原始字段名太长，先改成工作中更好读的字段名。
rename_cols = {
    "id": "id",
    "SeriousDlqin2yrs": "is_bad_customer",
    "RevolvingUtilizationOfUnsecuredLines": "credit_utilization",
    "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "past_due_30_59_count",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "open_credit_count",
    "NumberOfTimes90DaysLate": "past_due_90_count",
    "NumberRealEstateLoansOrLines": "real_estate_loan_count",
    "NumberOfTime60-89DaysPastDueNotWorse": "past_due_60_89_count",
    "NumberOfDependents": "dependent_count",
}

df = raw_df.rename(columns=rename_cols)

# timestamp 是公开镜像附加字段，不代表真实申请时间；后续不使用。
df = df.drop(columns=["timestamp"])

df.head()


,id,is_bad_customer,credit_utilization,age,past_due_30_59_count,debt_ratio,monthly_income,open_credit_count,past_due_90_count,real_estate_loan_count,past_due_60_89_count,dependent_count
0,1,1,0.7661,45,2,0.8030,"9,120.0000",13,0,6,0,2.0000
1,2,0,0.9572,40,0,0.1219,"2,600.0000",4,0,0,0,1.0000
2,3,0,0.6582,38,1,0.0851,"3,042.0000",2,1,0,0,0.0000
3,4,0,0.2338,30,0,0.0360,"3,300.0000",5,0,0,0,0.0000
4,5,0,0.9072,49,1,0.0249,"63,588.0000",7,0,1,0,0.0000


## 4. 样本规模和坏客户率


In [30]:
sample_count = len(df)
field_count = df.shape[1]
bad_count = df["is_bad_customer"].sum()
bad_rate = df["is_bad_customer"].mean()

basic_summary = pd.DataFrame({
    "指标": ["样本数", "字段数", "坏客户数", "坏客户率"],
    "结果": [sample_count, field_count, bad_count, bad_rate],
})

basic_summary


,指标,结果
0,样本数,"150,000.0000"
1,字段数,12.0000
2,坏客户数,"10,026.0000"
3,坏客户率,0.0668


In [33]:
target_summary = df["is_bad_customer"].value_counts().sort_index().reset_index()
target_summary.columns = ["is_bad_customer", "customer_count"]
target_summary["customer_type"] = target_summary["is_bad_customer"].map({0: "好客户", 1: "坏客户"})
target_summary["customer_rate"] = target_summary["customer_count"] / sample_count

target_summary


,is_bad_customer,customer_count,customer_type,customer_rate
0,0,139974,好客户,0.9332
1,1,10026,坏客户,0.0668


## 5. 字段角色

`customer_id` 只是记录编号，不能当作风险变量。`is_bad_customer` 是目标变量，其余字段是候选风险特征。


In [35]:
field_roles = pd.DataFrame({
    "字段": [
        "id",
        "is_bad_customer",
        "credit_utilization",
        "age",
        "past_due_30_59_count",
        "debt_ratio",
        "monthly_income",
        "open_credit_count",
        "past_due_90_count",
        "real_estate_loan_count",
        "past_due_60_89_count",
        "dependent_count",
    ],
    "角色": [
        "记录编号，不作为风险特征",
        "目标变量：未来两年是否严重逾期",
        "候选风险特征：额度使用率",
        "候选风险特征：年龄",
        "候选风险特征：30-59天逾期次数",
        "候选风险特征：负债率",
        "候选风险特征：月收入",
        "候选风险特征：开放信贷账户数",
        "候选风险特征：90天以上逾期次数",
        "候选风险特征：房地产贷款数",
        "候选风险特征：60-89天逾期次数",
        "候选风险特征：家属数量",
    ],
})

field_roles


,字段,角色
0,id,记录编号，不作为风险特征
1,is_bad_customer,目标变量：未来两年是否严重逾期
2,credit_utilization,候选风险特征：额度使用率
3,age,候选风险特征：年龄
4,past_due_30_59_count,候选风险特征：30-59天逾期次数
5,debt_ratio,候选风险特征：负债率
6,monthly_income,候选风险特征：月收入
7,open_credit_count,候选风险特征：开放信贷账户数
8,past_due_90_count,候选风险特征：90天以上逾期次数
9,real_estate_loan_count,候选风险特征：房地产贷款数


## 6. 关键数据质量提醒

这里不深入清洗，只列出下一步必须处理的问题。详细处理放到 `02_data_quality_cleaning.ipynb`。


In [36]:
income_missing_count = df["monthly_income"].isna().sum()
dependent_missing_count = df["dependent_count"].isna().sum()
age_zero_count = df["age"].eq(0).sum()
age_over_100_count = df["age"].gt(100).sum()
util_over_1_count = df["credit_utilization"].gt(1).sum()
open_credit_over_40_count = df["open_credit_count"].gt(40).sum()

past_due_has_96_98 = (
    df["past_due_30_59_count"].isin([96, 98])
    | df["past_due_60_89_count"].isin([96, 98])
    | df["past_due_90_count"].isin([96, 98])
)
past_due_96_98_count = past_due_has_96_98.sum()

quality_check = pd.DataFrame({
    "问题": [
        "月收入缺失",
        "家属数量缺失",
        "年龄为 0",
        "年龄大于 100",
        "额度使用率大于 1",
        "开户/贷款账户数大于 40",
        "逾期次数字段出现 96 或 98",
    ],
    "记录数": [
        income_missing_count,
        dependent_missing_count,
        age_zero_count,
        age_over_100_count,
        util_over_1_count,
        open_credit_over_40_count,
        past_due_96_98_count,
    ],
})

quality_check["占比"] = quality_check["记录数"] / sample_count
quality_check


,问题,记录数,占比
0,月收入缺失,29731,0.1982
1,家属数量缺失,3924,0.0262
2,年龄为 0,1,0.0000
3,年龄大于 100,13,0.0001
4,额度使用率大于 1,3321,0.0221
5,开户/贷款账户数大于 40,62,0.0004
6,逾期次数字段出现 96 或 98,269,0.0018


## 7. 本文件结论

- 数据可以继续用于风控分析；
- 坏客户率约为 6.68%，后续规则或模型都要和这个基准比较；
- `customer_id` 只用于定位记录，不作为风险特征；
- 月收入缺失、家属数量缺失、年龄异常、额度使用率异常、逾期特殊编码需要在 `02` 中处理。

下一步：进入 `02_data_quality_cleaning.ipynb`。
